In [2]:
import os
import pandas as pd
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/24/26 15:24:47] INFO     Using                                                                  ]8;id=584309;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=823347;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framewo                
                             rk\project\rich_logging.yml' as logging configuration.                                

[04/24/26 15:24:48] WARNING  c:\Users\User\miniconda3\envs\central\Lib\site-packages\requests\__ini ]8;id=587279;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=754182;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             t__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet                     
                             (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported                    
                             version!                                                                              
                               warnings.warn(                                                                      
                                                                                                                   

[04/24/26 15:24:51] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=753312;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=631450;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\CENTRAL\central-perm-flow


In [3]:
# Add the 'src' directory to the path
sys.path.append(os.path.abspath("src"))
import central_perm_flow.pipelines.data_processing.nodes as nodes_dproc
import central_perm_flow.pipelines.calac_activos_baj_grad.nodes as nodes_abg

# Importación de Fuentes

In [4]:
central_calaca_ext  = catalog.load('central_calendario_extendido')
central_calaca_uptoday  = catalog.load('central_calendario_extendido_uptoday')
central_estaca_sd = catalog.load('central_estaca_sd')

[04/24/26 15:24:52] INFO     Loading data from central_calendario_extendido                    ]8;id=418919;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=423941;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_calendario_extendido_uptoday            ]8;id=634697;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=155195;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

[04/24/26 15:24:53] INFO     Loading data from central_estaca_sd (ParquetDataset)...           ]8;id=445827;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=95144;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

## Importación de parámetros

In [5]:
parameters = catalog.load("parameters")

                    INFO     Loading data from parameters (MemoryDataset)...                   ]8;id=370894;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=247884;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [6]:
# bajas_calac
bajas_calac = parameters['bajas_calac']
graduados_calac = parameters['graduados_calac']

## Bajas

In [7]:
bajas_calendario_academico = nodes_abg.momento_baja(
    # 1. El DataFrame (Objeto en memoria)
    central_estaca_sd=central_estaca_sd, 
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 2. Valores extraídos de diccionarios de configuración
    central_col_fechadef=bajas_calac['central_col_fechadef'],
    central_col_fechatemp=parameters['bajas_calac']['central_col_fechatemp'],
    
    # 3. Nombre del dataset según el catálogo
    central_calaca= central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 4. Parámetros de transformación y cruce
    left_on=parameters['bajas_calac']['merge_left_on'],
    right_on=parameters['bajas_calac']['merge_right_on'],
    group_key=parameters['bajas_calac']['central_calaca_col_cohorte_inicial'],
    sort_cols=parameters['bajas_calac']['central_calaca_col_sort']
)

In [8]:
parameters['bajas_calac']['central_calaca_col_cohorte_inicial']

'fecha_ingreso'

In [9]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_desercion = (
    bajas_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico', 'fecha_ingreso', 'semana_acumulada', 'di']]
    .groupby(['cohorte_inicial','nivel_academico',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'di': 'sum'})
    .reset_index()
    .sort_values(by='di', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_desercion.style.bar(
    subset=['di'], 
    color='#f8766d', # Un rojo suave tipo "soft red"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,cohorte_inicial,nivel_academico,fecha_ingreso,semana_acumulada,di
10,2025 1b,posgrado,2025-05-12 00:00:00,3.000000,11
72,2026 1a,pregrado,2026-01-19 00:00:00,3.000000,11
78,2026 1b,posgrado,2026-03-16 00:00:00,1.000000,10
6,2025 1a,posgrado,2025-03-17 00:00:00,27.000000,10
32,2025 1c,posgrado,2025-07-07 00:00:00,23.000000,7
0,2025 1a,posgrado,2025-03-17 00:00:00,3.000000,5
69,2026 1a,posgrado,2026-01-19 00:00:00,3.000000,5
70,2026 1a,posgrado,2026-01-19 00:00:00,4.000000,5
62,2025 1e,pregrado,2025-10-27 00:00:00,8.000000,4
4,2025 1a,posgrado,2025-03-17 00:00:00,19.000000,4


In [10]:
bajas_calendario_academico = catalog.load('central_bajas_calendario_academico')

                    INFO     Loading data from central_bajas_calendario_academico              ]8;id=37018;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=917544;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

## Graduados

In [11]:
graduados_calendario_academico = nodes_abg.momento_grado(
    # 1. Referencias a datasets/DataFrames
    central_estaca=central_estaca_sd,
    central_calaca=central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
   
    col_gi=parameters['graduados_calac']['graduation_col_gi'],
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left=parameters['graduados_calac']['graduation_join_keys_left'],
    join_right=parameters['graduados_calac']['graduation_join_keys_right']
)

In [12]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_graduacion = (
    graduados_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico','nivel', 'fecha_ingreso', 'semana_acumulada', 'gi']]
    .groupby(['cohorte_inicial','nivel_academico','nivel',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'gi': 'sum'})
    .reset_index()
    .sort_values(by='gi', ascending=False)
    .head(100)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_graduacion.style.bar(
    subset=['gi'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,cohorte_inicial,nivel_academico,nivel,fecha_ingreso,semana_acumulada,gi
3,2025 1c,posgrado,especializacion,2025-07-07 00:00:00,32,58
2,2025 1b,posgrado,especializacion,2025-05-12 00:00:00,32,47
0,2025 1a,posgrado,especializacion,2025-03-17 00:00:00,32,24
1,2025 1a,posgrado,maestria,2025-03-17 00:00:00,48,16


## Activos

In [13]:
activos_calendario_academico = nodes_abg.momento_activos(
   central_estaca=central_estaca_sd,
    central_calaca=central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    col_di='di',
    col_gi='gi',
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    fallback_weeks=None,
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left='fecha_activo',
    join_right='fecha_inicio',
    group_key = 'fecha_ingreso'
) 

In [14]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_eng = (
    activos_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico','nivel','programa', 'fecha_ingreso', 'semana_acumulada','max_semana_teorica', 'engi']]
    .groupby(['cohorte_inicial','nivel_academico','nivel','programa',  'fecha_ingreso', 'semana_acumulada','max_semana_teorica'])
    .agg({'engi': 'sum'})
    .reset_index()
    .sort_values(by='engi', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_eng.style.bar(
    subset=['engi'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,cohorte_inicial,nivel_academico,nivel,programa,fecha_ingreso,semana_acumulada,max_semana_teorica,engi
3,2025 1c,posgrado,especializacion,auditoria y control,2025-07-07 00:00:00,32,32,4
0,2025 1a,posgrado,maestria,analitica de datos,2025-03-17 00:00:00,53,64,0
1,2025 1b,posgrado,maestria,analitica de datos,2025-05-12 00:00:00,46,64,0
2,2025 1b,posgrado,maestria,aseguramiento y auditoria de informacion,2025-05-12 00:00:00,46,48,0
4,2025 1c,posgrado,maestria,analitica de datos,2025-07-07 00:00:00,38,64,0
5,2025 1c,posgrado,maestria,aseguramiento y auditoria de informacion,2025-07-07 00:00:00,38,48,0
6,2025 1c,pregrado,pregrado,economia y finanzas,2025-07-07 00:00:00,38,128,0
7,2025 1c,pregrado,pregrado,ingenieria de sistemas y computacion,2025-07-07 00:00:00,38,128,0
8,2025 1d,posgrado,especializacion,auditoria y control,2025-09-01 00:00:00,30,32,0
9,2025 1d,posgrado,maestria,analitica de datos,2025-09-01 00:00:00,30,64,0


In [15]:
top_diez_activos = (
    activos_calendario_academico
    .groupby(['cohorte_inicial','fecha_ingreso','nivel_academico','nivel','programa',  'semana_acumulada','max_semana_teorica'])
    .agg(activos=('ai', 'sum')) # <-- Definimos: nombre = (columna, operacion)
    .reset_index()
    .sort_values(by='activos', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_activos.style.bar(
    subset=['activos'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,cohorte_inicial,fecha_ingreso,nivel_academico,nivel,programa,semana_acumulada,max_semana_teorica,activos
17,2025 1e,2025-10-27 00:00:00,pregrado,pregrado,derecho,22,144,92
28,2026 1a,2026-01-19 00:00:00,pregrado,pregrado,derecho,14,144,85
39,2026 1b,2026-03-16 00:00:00,pregrado,pregrado,derecho,6,144,74
19,2026 1a,2026-01-19 00:00:00,posgrado,especializacion,alta gerencia,14,32,59
4,2025 1c,2025-07-07 00:00:00,posgrado,maestria,analitica de datos,38,64,52
16,2025 1e,2025-10-27 00:00:00,pregrado,pregrado,contaduria publica,22,128,51
8,2025 1d,2025-09-01 00:00:00,posgrado,especializacion,auditoria y control,30,32,49
1,2025 1b,2025-05-12 00:00:00,posgrado,maestria,analitica de datos,46,64,48
32,2026 1b,2026-03-16 00:00:00,posgrado,especializacion,ciencias tributarias,6,32,47
21,2026 1a,2026-01-19 00:00:00,posgrado,especializacion,ciencias tributarias,14,32,47


In [16]:
def consolidar_universo_academico(
    df_bajas: pd.DataFrame,
    df_graduados: pd.DataFrame,
    df_activos: pd.DataFrame
) -> pd.DataFrame:
    """
    Concatena los tres estados académicos y normaliza indicadores.
    """
    # Unimos los tres dataframes
    universo = pd.concat([df_bajas, df_graduados, df_activos], ignore_index=True)
    
    # Normalizamos la columna engi: si es nula (viene de bajas/grados), es 0
    if 'engi' in universo.columns:
        universo['engi'] = universo['engi'].fillna(0).astype(int)
        
    return universo

In [17]:
df_bajas = catalog.load('central_bajas_calendario_academico')
df_graduados = catalog.load('central_graduados_calendario_academico')
df_activos = catalog.load('central_activos_calendario')

                    INFO     Loading data from central_bajas_calendario_academico              ]8;id=22798;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=42240;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_graduados_calendario_academico          ]8;id=233913;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=845790;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_activos_calendario (ParquetDataset)...  ]8;id=615290;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=201387;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [18]:
df_consolidado = catalog.load('central_estados_calac')

                    INFO     Loading data from central_estados_calac (ExcelDataset)...         ]8;id=738039;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=696371;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [19]:
df_consolidado['semana_acumulada']


0        3
1        3
2        3
3        3
4        3
        ..
1750    22
1751    22
1752    22
1753    22
1754     6
Name: semana_acumulada, Length: 1755, dtype: int64

In [20]:
top_diez_activos = (
    df_consolidado
    .groupby(['cohorte_inicial','fecha_ingreso'])
    .agg(estudiantes=('identificacion', 'count')) # <-- Definimos: nombre = (columna, operacion)
    .reset_index()
    .sort_values(by='estudiantes', ascending=False)
  
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_activos.style.bar(
    subset=['estudiantes'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook


In [21]:
top_diez_activos.sort_values(by= 'fecha_ingreso').head(20)

,cohorte_inicial,fecha_ingreso,estudiantes
0,2025 1a,2025-03-17,110
1,2025 1b,2025-05-12,155
2,2025 1c,2025-07-07,206
3,2025 1d,2025-09-01,132
4,2025 1e,2025-10-27,296
5,2026 1a,2026-01-19,456
6,2026 1b,2026-03-16,397


In [22]:
df_consolidado = catalog.load('central_estados_calac')

[04/24/26 15:24:55] INFO     Loading data from central_estados_calac (ExcelDataset)...         ]8;id=976427;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=546320;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [23]:
df_consolidado.groupby(['cohorte_inicial','fecha_ingreso']).agg({'ai':'sum',
                                                 'di':'sum',
                                                 'gi':'sum',
                                                 'engi':'sum',})

,,ai,di,gi,engi
cohorte_inicial,fecha_ingreso,,,,
2025 1a,2025-03-17,44,26,40,0
2025 1b,2025-05-12,73,35,47,0
2025 1c,2025-07-07,112,32,58,4
2025 1d,2025-09-01,121,11,0,0
2025 1e,2025-10-27,263,33,0,0
2026 1a,2026-01-19,426,30,0,0
2026 1b,2026-03-16,381,16,0,0


In [24]:
df_consolidado.columns


Index(['identificacion', 'codigo_sis', 'id_inconcert', 'nombres',
       'usuario_institucional', 'alianza', 'cohorte', 'cohorte_inicial',
       'cohorte_actual', 'nivel', 'nivel_academico', 'programa', 'estado',
       'bloque', 'descuentos', 'fecha_ingreso', 'fecha_de_registro',
       'fecha_de_baja_t', 'fecha_de_baja_d', 'fecha_baja',
       'fecha_de_reingreso', 'fecha_grado', 'fecha_activo', 'tipo_baja',
       'motivo_baja', 'submotivo_baja', 'comentarios', 'periodo_raw',
       'fecha_inicio', 'fecha_fin', 'shifted_fecha_inicio', 'semana',
       'semana_acumulada', 'month', 'mes_academico', 'anio_gregoriano',
       'mes_gregoriano', 'student_journey', 'tipo', 'max_semana_teorica',
       'exito_estudiantil', 'etapa_studen_journey', 'ai', 'di', 'gi', 'engi',
       'ci'],
      dtype='str')

In [25]:
df_consolidado.loc[:,['identificacion', 'codigo_sis', 'nombres',
       'usuario_institucional', 'alianza', 'cohorte', 'cohorte_inicial',
       'cohorte_actual', 'nivel', 'nivel_academico', 'programa', 'estado',
       'bloque', 'descuentos', 'fecha_ingreso', 'fecha_de_registro',
       'fecha_de_baja_t', 'fecha_de_baja_d', 'fecha_baja',
       'fecha_de_reingreso', 'fecha_grado', 'fecha_activo', 'tipo_baja',
       'motivo_baja', 'submotivo_baja', 'comentarios', 'periodo_raw']]

,identificacion,codigo_sis,nombres,usuario_institucional,alianza,cohorte,cohorte_inicial,cohorte_actual,nivel,nivel_academico,...,fecha_de_baja_d,fecha_baja,fecha_de_reingreso,fecha_grado,fecha_activo,tipo_baja,motivo_baja,submotivo_baja,comentarios,periodo_raw
0,80218031,9.912301e+10,henry aldemar zarrate hernandez,hzarrateh@ucentral.edu.co,central,2025-03-17,2025 1a,2025 1a,maestria,posgrado,...,2025-03-31,2025-03-31,NaT,NaT,NaT,confirmada,modalidad,dificultades adaptativas,estudiante quiere realizar su estudio pero en ...,periodo 1 2025 1a
1,52097052,9.912301e+10,olga lucia nontoa cristancho,onontoac@ucentral.edu.co,central,2025-03-17,2025 1a,2025 1a,especializacion,posgrado,...,2025-03-31,2025-03-31,NaT,NaT,NaT,confirmada,academico,inconformidad con contenido programatico,alumna queria cursar un programa de educacion ...,periodo 1 2025 1a
2,52361727,9.912301e+10,sandra paola baquero garcia,sbaquerog1@ucentral.edu.co,central,2025-03-17,2025 1a,2025 1a,especializacion,posgrado,...,2025-03-31,2025-03-31,NaT,NaT,NaT,confirmada,modalidad,cambio modalidad,alumna quiere cursar un programa de educacion ...,periodo 1 2025 1a
3,1014255653,9.912301e+10,mateo andres rodriguez morales,mrodriguezm@ucentral.edu.co,central,2025-03-17,2025 1a,2025 1a,maestria,posgrado,...,NaT,2025-03-31,NaT,NaT,NaT,confirmada,academico,desde ventas,estudiante quien tuvo ingreso tardio aun pagan...,periodo 1 2025 1a
4,79671835,9.912301e+10,william rene albornoz zoste,walbornozz@ucentral.edu.co,central,2025-03-17,2025 1a,2025 1a,maestria,posgrado,...,2025-03-31,2025-03-31,NaT,NaT,NaT,confirmada,declina,desde ventas,camilo andres diaz segura me oriento en el mes...,periodo 1 2025 1a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1750,53003163,9.912301e+10,jessica paola bautista alvarez,jbautistaa1@ucentral.edu.co,central,2025-10-27,2025 1e,2025 2e,pregrado,pregrado,...,NaT,NaT,NaT,NaT,2026-04-24,NaN,NaN,NaN,NaN,periodo 2 2025 2e
1751,53001482,9.912301e+10,sandra milena vargas santamaria,svargass5@ucentral.edu.co,central,2025-10-27,2025 1e,2025 2e,pregrado,pregrado,...,NaT,NaT,NaT,NaT,2026-04-24,NaN,NaN,NaN,NaN,periodo 2 2025 2e
1752,52957468,9.912301e+10,yolanda daza alvarado,ydazaa@ucentral.edu.co,central,2025-10-27,2025 1e,2025 2e,pregrado,pregrado,...,NaT,NaT,NaT,NaT,2026-04-24,NaN,NaN,NaN,NaN,periodo 2 2025 2e
1753,1019066326,9.912301e+10,vanessa buitrago garzon,vgarzong1@ucentral.edu.co,central,2025-10-27,2025 1e,2025 2e,pregrado,pregrado,...,NaT,NaT,NaT,NaT,2026-04-24,NaN,NaN,NaN,NaN,periodo 2 2025 2e


# Solicitudes

## Activos

In [26]:
central_activos_calendario = catalog.load('central_activos_calendario')
# Pregrado
mask_pregrado = central_activos_calendario['nivel_academico'] == 'pregrado'
central_activos_calendario_pregrado = central_activos_calendario[mask_pregrado]
# Posgrado
mask_posgrado = central_activos_calendario['nivel_academico'] == 'posgrado'
central_activos_calendario_posgrado = central_activos_calendario[mask_posgrado]

[04/24/26 15:24:56] INFO     Loading data from central_activos_calendario (ParquetDataset)...  ]8;id=257651;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=557119;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [27]:
comunicaciones_col = ['id_inconcert','nombres','identificacion','usuario_institucional','programa','cohorte_actual']

In [28]:
central_activos_calendario_pregrado_comunicaciones = central_activos_calendario_pregrado.loc[:,comunicaciones_col].rename(columns = {'id_inconcert':'ID Inconcert',
                                                                                              'nombres':'Nombre Completo',
                                                                                              'identificacion': 'Documento de identidad',
                                                                                              'usuario_institucional':'Correo electrónico institucional',
                                                                                              'programa':'Programa académico',
                                                                                              'cohorte_actual':'Periodo Academico'})
central_activos_calendario_pregrado_comunicaciones['Telefono'] = None
central_activos_calendario_pregrado_comunicaciones['Correo electrónico personal'] = None

In [29]:
central_activos_calendario_prosgrado_comunicaciones = central_activos_calendario_posgrado.loc[:,comunicaciones_col].rename(columns = {'id_inconcert':'ID Inconcert',
                                                                                              'nombres':'Nombre Completo',
                                                                                              'identificacion': 'Documento de identidad',
                                                                                              'usuario_institucional':'Correo electrónico institucional',
                                                                                              'programa':'Programa académico',
                                                                                              'cohorte_actual':'Periodo Academico'})
central_activos_calendario_prosgrado_comunicaciones['Telefono'] = None
central_activos_calendario_prosgrado_comunicaciones['Correo electrónico personal'] = None

In [30]:
comunicaciones_col_output = ['ID Inconcert', 'Nombre Completo', 'Documento de identidad',
                        'Correo electrónico personal','Correo electrónico institucional','Telefono', 'Programa académico',
                        'Periodo Academico']

# Pregrado

In [31]:
central_activos_calendario_pregrado_comunicaciones.shape

(582, 8)

In [32]:
central_activos_calendario_pregrado_comunicaciones.loc[:,comunicaciones_col_output].to_excel('data/07_model_output/01_solicitudes/central_activos_pregrado.xlsx', index = False)

# Posgrado

In [33]:
central_activos_calendario_prosgrado_comunicaciones.loc[:,comunicaciones_col_output].to_excel('data/07_model_output/01_solicitudes/central_activos_prosgrado.xlsx', index = False)

In [34]:
central_activos_calendario_prosgrado_comunicaciones.shape

(842, 8)

In [39]:
central_bajas_calendario_academico = catalog.load('central_bajas_calendario_academico')

[04/24/26 15:25:41] INFO     Loading data from central_bajas_calendario_academico              ]8;id=224390;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=164232;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

In [49]:
mask_cohorte = central_bajas_calendario_academico['cohorte'] == pd.to_datetime('2025-09-01')
mask_programa = central_bajas_calendario_academico['programa'] == 'ingenieria de sistemas y computacion'

In [50]:
central_bajas_calendario_academico.loc[mask_cohorte & mask_programa,:]

,identificacion,codigo_sis,id_inconcert,nombres,usuario_institucional,alianza,cohorte,fecha_ingreso,fecha_de_registro,nivel,...,fecha_fin,shifted_fecha_inicio,semana,semana_acumulada,month,mes_academico,anio_gregoriano,mes_gregoriano,student_journey,tipo
95,1024553877,9.912301e+10,88032.0,nathalia romero acuna,nromeroa3@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,2025-09-14,2025-09-15,2.0,2.0,1.0,m1,2025.0,9.0,onboarding,ingreso 1
97,1019056963,9.912301e+10,69561.0,jonathan parra ajiaco,jparraa@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,2025-10-05,2025-10-06,5.0,5.0,2.0,m2,2025.0,10.0,onboarding,ingreso 1
100,1023887774,9.912301e+10,87283.0,carlos arturo rios sanchez,crioss1@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,2025-12-21,2026-01-19,8.0,16.0,4.0,m4,2025.0,12.0,q1,ingreso 1


In [44]:
mask_cohorte = central_activos_calendario['cohorte'] == pd.to_datetime('2025-09-01')
mask_programa = central_activos_calendario['programa'] == 'ingenieria de sistemas y computacion'

In [47]:
central_activos_calendario[mask_cohorte & mask_programa]

,identificacion,codigo_sis,id_inconcert,nombres,usuario_institucional,alianza,cohorte,fecha_ingreso,fecha_de_registro,nivel,...,semana,semana_acumulada,month,mes_academico,anio_gregoriano,mes_gregoriano,student_journey,tipo,engi,ai
824,1193201129,9.912301e+10,84602.0,angel fernando toscano rodriguez,atoscanor@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,6,30,8,m8,2026,4,qa,ciclo academico 2,0,1
825,1120475167,9.912301e+10,88006.0,yeferson coquinche romero,ycoquincher@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,6,30,8,m8,2026,4,qa,ciclo academico 2,0,1
826,1073164906,9.912301e+10,87339.0,yarleidy areiza tobon,yareizat@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,6,30,8,m8,2026,4,qa,ciclo academico 2,0,1
827,1033705986,9.912301e+10,70326.0,sergio sebastian cante aguirre,scantea@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,6,30,8,m8,2026,4,qa,ciclo academico 2,0,1
828,1030526416,9.912301e+10,32266.0,jessyca natalia grandas vasquez,jgrandasv@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,6,30,8,m8,2026,4,qa,ciclo academico 2,0,1
829,1025532503,9.912301e+10,31191.0,miguel angel uribe franco,muribef@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,6,30,8,m8,2026,4,qa,ciclo academico 2,0,1
830,1023978428,9.912301e+10,16134.0,juan jose cruz lara,jcruzl2@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,6,30,8,m8,2026,4,qa,ciclo academico 2,0,1
831,1023365019,9.912301e+10,31382.0,julian samuel arango mahecha,jarangom1@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,6,30,8,m8,2026,4,qa,ciclo academico 2,0,1
833,1018488606,9.912301e+10,85827.0,luis enrique jimenez perilla,ljimenezp5@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,6,30,8,m8,2026,4,qa,ciclo academico 2,0,1
834,1014254303,9.912301e+10,20455.0,jhojan camilo forero carvajal,jforeroc11@ucentral.edu.co,central,2025-09-01,2025-09-01,2026-04-10,pregrado,...,6,30,8,m8,2026,4,qa,ciclo academico 2,0,1
